In [80]:
"""
Query Expansion RAG Pipeline -- Production-Grade
=================================================
Architecture   : FIVE configurations covering every query expansion pattern:
                 (1) Standard RAG Baseline            -- single query, zero expansion
                 (2) MultiQueryRetriever (Built-In)   -- LangChain from_llm, 3 perspectives
                 (3) Custom Perspective Expansion     -- 5 views: synonym, expert, narrow,
                                                         broad, adversarial + RRF fusion
                 (4) Step-Back + Original Fusion      -- abstraction query + original RRF
                 (5) RFG: Retrieval-Feedback-Grounded -- pseudo-relevance feedback loop,
                                                         synonym/acronym expansion [BEST]

Vector Store   : ChromaDB  (persistent, metadata filtering, MMR support)
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query instruction prefix)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+  (MultiQueryRetriever, LCEL, EnsembleRetriever)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
What Query Expansion Is and Why It Exists
=============================================================================
Distance-based similarity search retrieves the K document chunks whose
embedding vectors are closest to the query embedding vector. This design
has a structural limitation: it finds documents that match the VOCABULARY
AND PHRASING of the query, not necessarily the underlying INFORMATION NEED.

THREE ROOT CAUSES OF RECALL FAILURE IN STANDARD RAG:
    1. VOCABULARY MISMATCH: The user asks about "attention mechanism" but
       the relevant chunk says "scaled dot-product self-attention computation".
       Neither phrase subsumes the other, but they mean the same thing.
       The cosine distance between their embeddings is non-trivial.

    2. PERSPECTIVE ASYMMETRY: A user unfamiliar with a field phrases a query
       from a learner's perspective ("how does attention work") while the indexed
       text is written from an expert's perspective ("the attention function
       maps a query and a set of key-value pairs to an output"). Both are about
       the same content but embed to different vector regions.

    3. GRANULARITY MISMATCH: A broad query ("transformer training") misses
       narrow but relevant chunks ("WMT 2014 English-to-German translation
       with 6 encoder layers"). A narrow query ("WMT 2014 EN-DE BLEU") misses
       the broader architectural context. The optimal retrieval covers both.

WHAT QUERY EXPANSION DOES:
    Query expansion consists of asking an LLM to produce a number of
    similar queries to a user query. We are then able to use each of these
    queries in the retrieval process, increasing the number and relevance
    of retrieved documents.

    Each expanded query targets a DIFFERENT region of the embedding space:
        Query 1 (synonym):   "self-attention scoring function"
        Query 2 (expert):    "scaled dot-product query-key-value attention"
        Query 3 (narrow):    "attention weight softmax computation"
        Query 4 (broad):     "transformer model architecture components"
        Query 5 (original):  "how does attention work"

    Documents matching any of these queries are pooled and deduplicated,
    dramatically increasing recall while deduplication preserves precision.

=============================================================================
The Five Query Expansion Strategies
=============================================================================

(1) BASELINE: Single query. Zero expansion. Establishes recall floor.

(2) MULTI-QUERY (Built-In): Given a query, use an LLM to write a set of
    queries. Retrieve docs for each query. Return the unique union of all
    retrieved docs. LangChain's MultiQueryRetriever.from_llm() automates
    this: generates 3 perspective variants, retrieves for each, deduplicates.

(3) CUSTOM PERSPECTIVE EXPANSION: 5 explicitly-typed expansions -- synonym
    rewrite, expert-vocabulary rewrite, narrow scope, broad scope, adversarial
    (what would the opposite query retrieve, to identify missing coverage).
    Results fused via Reciprocal Rank Fusion (RRF) with equal weights.

(4) STEP-BACK PROMPTING: This paper uses an LLM to generate a "step back"
    question. With retrieval, both the "step back" question and the original
    question are used to do retrieval, and then both results are used to
    ground the language model response. The step-back query retrieves
    PRINCIPLE-LEVEL context (broader concepts) that grounds the specific answer.

(5) RFG (Retrieval-Feedback-Grounded) EXPANSION: One of the main problems
    with multi-query approaches is that they depend on the parametric knowledge
    of the model. When the domain is very different from the one on which the
    LLM was trained, generating multiple queries tends to hallucinate. The RFG
    framework uses pseudo-relevance feedback: an initial retrieval step retrieves
    documents, and pseudo-queries are generated based on the retrieved information,
    significantly improving retrieval by avoiding hallucinations.
    Config 5 adds synonym and acronym expansion on top of the feedback loop.

=============================================================================
The Synonym/Acronym Expansion Problem
=============================================================================
Acronyms and abbreviations are common in queries and documents. "LLM"
should ideally match documents mentioning "Large Language Model." A system
maintains a mapping for these. While powerful, naive synonym expansion can
lead to query drift, where the expanded query loses its original focus.
Careful selection or weighting of expanded terms is necessary.

In Config 5, synonyms and acronyms are:
    - Extracted from the initial retrieved documents (not hallucinated).
    - Used to AUGMENT the reformulated query (not replace it).
    - Bounded to technical terms found in the actual corpus.
This grounds expansion in retrieved evidence, preventing drift.

"""


'\nQuery Expansion RAG Pipeline -- Production-Grade\n=================================================\nArchitecture   : FIVE configurations covering every query expansion pattern:\n                 (1) Standard RAG Baseline            -- single query, zero expansion\n                 (2) MultiQueryRetriever (Built-In)   -- LangChain from_llm, 3 perspectives\n                 (3) Custom Perspective Expansion     -- 5 views: synonym, expert, narrow,\n                                                         broad, adversarial + RRF fusion\n                 (4) Step-Back + Original Fusion      -- abstraction query + original RRF\n                 (5) RFG: Retrieval-Feedback-Grounded -- pseudo-relevance feedback loop,\n                                                         synonym/acronym expansion [BEST]\n\nVector Store   : ChromaDB  (persistent, metadata filtering, MMR support)\nEmbeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query instruction prefix)\nLLM            : Azure OpenAI

In [81]:

import os
import re
import sys
import time
import pickle
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

# ---------------------------------------------------------------------------
# pip install langchain langchain-community langchain-ollama langchain-chroma
#             chromadb sentence-transformers pypdf lark
# ---------------------------------------------------------------------------
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import BaseOutputParser
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [82]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import EnsembleRetriever , ContextualCompressionRetriever

In [83]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("query_expansion_rag")

# Enable MultiQueryRetriever verbose logging (shows generated queries)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [84]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class QEConfig:
    """
    Centralized configuration for the Query Expansion RAG pipeline.

    Expansion Temperature Design (LLM_EXPANSION_TEMPERATURE = 0.3):
        Query expansion needs ENOUGH variation to produce vocabulary-diverse
        reformulations but NOT so much variation that queries drift away from
        the original information need. Temperature=0.3 is the production sweet spot:
            - 0.0: deterministic, all reformulations are too similar.
            - 0.3: light variation -- different synonyms, reordered clauses.
            - 0.7: high variation -- risks topic drift, hallucinated expansions.
            - 1.0: not suitable -- queries often become off-topic or semantically wrong.

        Step-Back prompting uses temperature=0.0 to ensure the abstraction step
        is consistent and reliable (the step-back question should be the same
        every time for the same input).

    N_EXPANSIONS (5 for Config 3):
        Five explicitly-typed expansion views provide maximum coverage:
            synonym:     Different vocabulary, same specificity.
            expert:      Technical/domain-expert phrasing.
            narrow:      Specific sub-question, higher precision.
            broad:       Generalized parent question, higher recall.
            adversarial: Contrasting/opposite framing -- surfaces documents
                         that discuss both sides of a comparison.
        Each view targets a distinct region of the embedding manifold.

    RFG_INITIAL_K (10):
        The initial retrieval in RFG captures the "pseudo-relevant" documents
        that anchor query reformulation. More documents = more vocabulary signal
        for expansion = better synonym/acronym extraction.
        Trade-off: More initial docs = more LLM context = slower expansion generation.
        10 is the validated production default for research paper corpora.

    DEDUP_PREFIX_LEN (200):
        Deduplication across multiple expanded queries uses the first 200 chars
        of page_content as the document identity key. This is robust to minor
        whitespace differences between the same chunk retrieved via different queries.
        Longer prefixes (500+) risk missing near-duplicate chunks with different
        leading whitespace. Shorter prefixes (50) risk false positive deduplication
        for chunks that share opening phrases but differ in content.

    RRF_K (60):
        Reciprocal Rank Fusion constant. RRF score = sum_over_retrievers 1/(k + rank_i).
        k=60 is the validated default from the RRF paper (Cormack et al. 2009).
        It smooths the rank contribution: position 1 scores 1/61=0.016,
        position 10 scores 1/70=0.014. The difference between rank 1 and 10
        is small, making RRF robust to ordering noise within individual retrievers.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- ChromaDB ---------------------------------------------------------
    CHROMA_PERSIST_DIR:     str = "./chroma_qe_db"
    CHROMA_COLLECTION_NAME: str = "qe_research_papers"

    # --- BGE Bi-Encoder ---------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Retrieval K parameters -------------------------------------------
    SEARCH_K:       int = 4    # final docs per expanded query for retrieval
    FINAL_K:        int = 4    # docs passed to LLM after dedup/fusion
    RFG_INITIAL_K:  int = 10   # initial retrieval K for pseudo-relevance feedback

    # --- Expansion Parameters --------------------------------------------
    N_EXPANSIONS:                  int   = 5    # Config 3: typed perspectives
    LLM_EXPANSION_TEMPERATURE:     float = 0.3  # expansion generation (creative)
    LLM_STEPBACK_TEMPERATURE:      float = 0.0  # step-back (deterministic)
    LLM_ANSWER_TEMPERATURE:        float = 0.0  # answer generation (deterministic)

    # --- RRF Parameters --------------------------------------------------
    RRF_K:         int   = 60
    DEDUP_PREFIX_LEN: int = 200

    # --- Text Splitting ---------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Azure OpenAI LLM ------------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int            = 1024

    # --- RAG Answer Prompt -----------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved using query expansion):
{context}

Question: {question}

Precise technical answer:"""



In [85]:

# ===========================================================================
# SECTION 2 -- QUERY EXPANSION TRACE
# ===========================================================================

@dataclass
class QETrace:
    """
    Records the complete query expansion execution trace for observability.

    Fields:
        original_query   : Raw user input.
        strategy         : Config label.
        expanded_queries : All queries generated (including original if applicable).
        docs_per_query   : Number of docs retrieved for each expanded query.
        total_candidates : Total docs before deduplication.
        final_docs       : Docs after deduplication/fusion passed to LLM.
        final_answer     : LLM-generated answer.
        expansion_ms     : Time for all LLM query generation calls.
        retrieval_ms     : Time for all retrieval operations.
        total_ms         : End-to-end wall clock.

    The expansion_ms/retrieval_ms split is the key production latency metric.
    In production, if expansion_ms > 400ms, consider:
        - Using a smaller/faster model for expansion (GPT-4o-mini vs GPT-4o).
        - Caching expanded queries for repeated input patterns.
        - Pre-computing expansions offline for known query templates.
    """
    original_query:   str
    strategy:         str
    expanded_queries: List[str]       = field(default_factory=list)
    docs_per_query:   List[int]       = field(default_factory=list)
    total_candidates: int             = 0
    final_docs:       List[Document]  = field(default_factory=list)
    final_answer:     str             = ""
    expansion_ms:     float           = 0.0
    retrieval_ms:     float           = 0.0
    total_ms:         float           = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*70}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.original_query[:80]}")
        print(f"{'='*70}")

        if self.expanded_queries:
            print(f"\n  Expanded Queries ({len(self.expanded_queries)}):")
            for i, q in enumerate(self.expanded_queries, 1):
                n = self.docs_per_query[i-1] if i <= len(self.docs_per_query) else "?"
                print(f"  [{i}] ({n} docs)  {q}")

        print(f"\n  Candidates: {self.total_candidates}  ->  Final: {len(self.final_docs)}")
        print(f"  Sources: ", end="")
        for doc in self.final_docs:
            paper = doc.metadata.get("paper_name", "?")[:25]
            page  = doc.metadata.get("page", "?")
            print(f"[{paper} p{page}]", end=" ")
        print()

        print(f"\n  Latency: expansion={self.expansion_ms:.1f}ms  "
              f"retrieval={self.retrieval_ms:.1f}ms  "
              f"total={self.total_ms:.1f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:400]}")
        print("=" * 70 + "\n")



In [86]:

# ===========================================================================
# SECTION 3 -- PDF DOWNLOADER AND CHUNKER
# ===========================================================================

def download_pdfs(config: QEConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}'. Place at '{dest}'.") from exc

    return paths



In [87]:


def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: QEConfig,
) -> List[Document]:
    """Load PDFs and split into chunks with paper_name and page metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})

        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)

    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks



In [88]:

def build_bge_embeddings(config: QEConfig) -> HuggingFaceEmbeddings:
    """Initialize BGE-large-en-v1.5 bi-encoder."""
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


In [89]:

def build_chroma_index(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: QEConfig,
) -> Chroma:
    """
    Build or load ChromaDB vector store.

    ChromaDB is used here instead of FAISS because:
        - ChromaDB's MMR search is used in the base retriever for Config 2/3
          to avoid retrieving near-duplicate chunks for the same expanded query.
        - ChromaDB's metadata filtering could be combined with expansion for
          domain-scoped expansion (out of scope here but architecture-ready).
    """
    # Some notebook environments leak user-site packages ahead of conda env packages,
    # which can break chromadb/opentelemetry imports. Force env-local resolution first.
    try:
        import site
        user_site = site.getusersitepackages()
        if user_site in sys.path:
            sys.path.remove(user_site)
    except Exception:
        pass

    # If opentelemetry was already imported from user-site, clear it before importing chromadb.
    for mod in list(sys.modules):
        if mod.startswith("opentelemetry") and "chromadb" not in mod:
            del sys.modules[mod]

    try:
        import chromadb
        client = chromadb.PersistentClient(path=config.CHROMA_PERSIST_DIR)
        col    = client.get_or_create_collection(config.CHROMA_COLLECTION_NAME)
        if col.count() > 0:
            logger.info(
                "Loading ChromaDB from '%s' (%d vectors) ...",
                config.CHROMA_PERSIST_DIR, col.count(),
            )
            return Chroma(
                client=client,
                collection_name=config.CHROMA_COLLECTION_NAME,
                embedding_function=embeddings,
                persist_directory=config.CHROMA_PERSIST_DIR,
            )
    except Exception as exc:
        logger.warning("ChromaDB load check failed (%s), rebuilding ...", exc)

    logger.info("Building ChromaDB from %d chunks ...", len(chunks))
    t0 = time.perf_counter()
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=config.CHROMA_PERSIST_DIR,
        collection_name=config.CHROMA_COLLECTION_NAME,
    )
    logger.info(
        "ChromaDB built. Vectors: %d  (%.2fs)",
        vs._collection.count(), time.perf_counter() - t0,
    )
    return vs


In [90]:

def build_ollama_llm(
    config: QEConfig,
    temperature: float = None,
) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
    return ChatOllama(
        base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
        model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


In [91]:

# ===========================================================================
# SECTION 5 -- SHARED UTILITIES
# ===========================================================================

def dedup_union(
    doc_lists: List[List[Document]],
    prefix_len: int = 200,
) -> List[Document]:
    """
    Deduplicate and union multiple document lists using content prefix as key.

    Documents are added in FIRST-SEEN order. If the same chunk is retrieved
    by queries 1, 3, and 5, it appears once in the output at the position
    it was first encountered (from query 1). This preserves the ordering
    semantics: a document retrieved by multiple expanded queries has higher
    expected relevance than one retrieved by only one query.

    Args:
        doc_lists  : List of lists of documents (one list per expanded query).
        prefix_len : Characters of page_content used as identity key.

    Returns:
        Flat deduplicated list of Documents.
    """
    seen:   Dict[str, bool] = {}
    result: List[Document]  = []
    for docs in doc_lists:
        for doc in docs:
            key = doc.page_content.strip()[:prefix_len]
            if key not in seen:
                seen[key]  = True
                result.append(doc)
    return result


In [92]:


def rrf_fusion(
    doc_lists:   List[List[Document]],
    weights:     Optional[List[float]] = None,
    k:           int = 60,
    final_k:     int = 4,
    prefix_len:  int = 200,
) -> List[Document]:
    """
    Reciprocal Rank Fusion over multiple retrieved document lists.

    RRF formula:  score(doc) = sum_i  w_i / (k + rank_i(doc))
    where rank_i is the 1-based rank of doc in retriever i's results,
    w_i is the weight for retriever i (default: 1/N equal weights),
    and k=60 is the smoothing constant.

    Documents not appearing in retriever i's results receive rank=infinity
    (zero contribution from that retriever to the RRF score).

    This is the standard fusion method for multi-query expansion because:
        - It is SCALE-INVARIANT: similarity scores from different expanded
          queries are not comparable (different query distributions), but
          ranks are always on [1, K] scale.
        - It is ROBUST TO NOISE: a document at rank 5 vs rank 6 is nearly
          identical in RRF score, preventing minor ranking noise from dominating.
        - It REWARDS CONSISTENCY: a document appearing at rank 3 in ALL five
          expanded queries scores much higher than one at rank 1 in only one.

    Args:
        doc_lists  : Per-query retrieved document lists.
        weights    : Per-query weight (default: equal 1/N).
        k          : RRF smoothing constant (default 60).
        final_k    : Number of top documents to return.
        prefix_len : Identity key length for deduplication.

    Returns:
        Top-final_k documents by RRF score.
    """
    N = len(doc_lists)
    if weights is None:
        weights = [1.0 / N] * N
    assert len(weights) == N, "weights must match number of doc_lists"

    # Build {doc_key -> Document} index and RRF score accumulator
    doc_index: Dict[str, Document] = {}
    scores:    Dict[str, float]    = {}

    for retriever_idx, docs in enumerate(doc_lists):
        w = weights[retriever_idx]
        for rank, doc in enumerate(docs, start=1):
            key = doc.page_content.strip()[:prefix_len]
            if key not in doc_index:
                doc_index[key] = doc
                scores[key]    = 0.0
            scores[key] += w / (k + rank)

    # Sort by RRF score descending and return top final_k
    sorted_keys = sorted(scores, key=lambda kk: scores[kk], reverse=True)
    return [doc_index[kk] for kk in sorted_keys[:final_k]]


def format_docs(docs: List[Document]) -> str:
    """Format documents into annotated context block for LLM."""
    parts = []
    for i, doc in enumerate(docs, 1):
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = doc.metadata.get("page", "?")
        chars = len(doc.page_content)
        parts.append(
            f"[Rank {i} | {paper} | Page {page} | {chars} chars]\n"
            f"{doc.page_content.strip()}"
        )
    return ("\n\n" + "-" * 60 + "\n\n").join(parts)


In [93]:

# ===========================================================================
# SECTION 6 -- FIVE QUERY EXPANSION CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Standard RAG Baseline
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question:   str,
    vectorstore: Chroma,
    llm_answer: ChatOllama,
    config:     QEConfig,
) -> QETrace:
    """
    Configuration 1: Standard RAG Baseline (no query expansion).

    Single query, direct embedding, ChromaDB similarity search, LLM answer.
    Included as the control condition to measure expansion improvement.

    ChromaDB `search_type="mmr"` with `lambda_mult=0.5` provides Maximum
    Marginal Relevance at the retrieval level: retrieved docs are diverse
    (not all near-duplicate chunks from the same section). This is important
    so the baseline is fair -- we do not give the expanded configs diversity
    for free that the baseline does not also have.
    """
    trace = QETrace(original_query=question, strategy="Config1_Baseline")
    t0    = time.perf_counter()

    trace.expanded_queries = [question]

    t_ret  = time.perf_counter()
    docs   = vectorstore.similarity_search(question, k=config.FINAL_K)
    trace.retrieval_ms   = (time.perf_counter() - t_ret) * 1000
    trace.docs_per_query = [len(docs)]
    trace.total_candidates = len(docs)
    trace.final_docs     = docs

    context = format_docs(docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    answer  = (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [94]:


# ---------------------------------------------------------------------------
# CONFIG 2: MultiQueryRetriever -- Built-In LangChain
# ---------------------------------------------------------------------------

def run_config2_multi_query_builtin(
    question:    str,
    vectorstore: Chroma,
    llm_expand:  ChatOllama,
    llm_answer:  ChatOllama,
    config:      QEConfig,
) -> QETrace:
    """
    Configuration 2: LangChain MultiQueryRetriever with include_original=True.

    MultiQueryRetriever.from_llm() builds the following pipeline internally:
        prompt  -> llm_expand -> LineListOutputParser -> List[str] (3 queries)
        Then for each query: base_retriever.invoke(query) -> List[Document]
        Finally: unique_union of all retrieved docs (dedup by page_content hash)

    Key parameters:
        include_original=True:
            The original user question is added to the generated queries list,
            ensuring the original query's retrieved documents are always included.
            Without this, if the LLM generates 3 reformulations that all miss
            the original phrasing, the original query's results are absent.
            Always use include_original=True in production.

        verbose=True (via logging.getLogger):
            MultiQueryRetriever logs generated queries at INFO level when
            logging.getLogger("langchain.retrievers.multi_query") is set to INFO.
            This provides free observability into what queries were generated.
            Enabled at the top of this module.

        base_retriever (MMR, fetch_k=15):
            Each expanded query retrieves via MMR to ensure diversity WITHIN
            each query's result set. Without MMR, the top-K from a single
            expanded query may all be near-duplicate chunks from the same section,
            which adds noise to the multi-query union.

    Default MultiQueryRetriever prompt generates 3 perspective variants:
        "Your task is to generate 3 different versions of the given user
        question to retrieve relevant documents from a vector database.
        By generating multiple perspectives on the user question, your goal
        is to help the user overcome some of the limitations of distance-based
        similarity search."

    The unique union across 3 queries + original = up to 4 * SEARCH_K candidates,
    deduplicated to typically 6-12 unique documents for final_k=4 trimming.

    Args:
        question    : User query.
        vectorstore : ChromaDB.
        llm_expand  : LLM with expansion temperature=0.3.
        llm_answer  : LLM with answer temperature=0.0.
        config      : QEConfig.

    Returns:
        QETrace with 3+1 expanded queries and unique union retrieval.
    """
    trace = QETrace(original_query=question, strategy="Config2_MultiQuery_BuiltIn")
    t0    = time.perf_counter()

    # Base retriever: MMR for intra-query diversity
    base_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": config.SEARCH_K, "fetch_k": config.SEARCH_K * 3, "lambda_mult": 0.5},
    )

    # MultiQueryRetriever: generates 3 variants + includes original
    t_expand = time.perf_counter()
    mq_retriever = MultiQueryRetriever.from_llm(
        retriever=base_retriever,
        llm=llm_expand,
        include_original=True,   # ALWAYS True in production
    )
    trace.expansion_ms = (time.perf_counter() - t_expand) * 1000

    # Single invoke -- internally retrieves for each generated query + original
    t_ret  = time.perf_counter()
    docs   = mq_retriever.invoke(question)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    # MultiQueryRetriever hides per-query counts; approximate as total/4
    approx_n = max(1, len(docs))
    trace.expanded_queries = [
        "[3 LLM-generated variants + original query]",
        "(exact queries logged via langchain.retrievers.multi_query logger)",
    ]
    trace.docs_per_query   = [approx_n]
    trace.total_candidates = len(docs)
    trace.final_docs       = docs[: config.FINAL_K]

    context = format_docs(trace.final_docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    answer  = (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [95]:

# ---------------------------------------------------------------------------
# CONFIG 3: Custom 5-Perspective Expansion with RRF Fusion
# ---------------------------------------------------------------------------

PERSPECTIVE_EXPANSION_PROMPT = """You are an expert in retrieval and information science.
Given the original question below, generate EXACTLY {n_views} reformulations covering these
specific retrieval perspectives. Output ONE question per line, no numbering, no labels.

Perspective order (one question each):
1. SYNONYM REWRITE: Same question using alternative vocabulary and synonyms
2. EXPERT VOCABULARY: Rephrase using domain-expert technical terminology
3. NARROW SCOPE: A more specific sub-question targeting a precise technical detail
4. BROAD SCOPE: A more general parent question providing wider context
5. ADVERSARIAL: What contrasting or comparative question would surface related content?

Output exactly {n_views} lines, each a complete standalone question.

Original question: {question}

{n_views} reformulations:"""


def run_config3_perspective_expansion(
    question:    str,
    vectorstore: Chroma,
    llm_expand:  ChatOllama,
    llm_answer:  ChatOllama,
    config:      QEConfig,
) -> QETrace:
    """
    Configuration 3: Five Explicitly-Typed Perspective Expansions + RRF Fusion.

    Unlike MultiQueryRetriever (Config 2) which generates 3 generic variants,
    Config 3 generates 5 TYPED perspectives, each targeting a different recall
    failure mode:

        SYNONYM REWRITE:
            Target: Vocabulary mismatch failure. The document uses different
            words for the same concept as the query. Synonym rewrite covers
            the alternate vocabulary space.
            Example: "attention mechanism" -> "query-key-value scoring function"

        EXPERT VOCABULARY:
            Target: Expertise-level mismatch. User phrases query at a learner
            level; documents are written for experts. Expert rewrite matches
            the document vocabulary distribution.
            Example: "how attention works" -> "scaled dot-product attention
            computation over Q K V projections"

        NARROW SCOPE:
            Target: Over-broad query. Query retrieves many loosely related chunks
            but misses the specific passage containing the exact answer. Narrow
            scope drills into the specific fact needed.
            Example: "transformer attention" -> "softmax normalization temperature
            in scaled dot-product attention formula"

        BROAD SCOPE:
            Target: Over-specific query. Query is too narrow and misses related
            context that frames the answer. Broad scope retrieves the section
            headers, introductions, and overview passages.
            Example: "attention weight formula" -> "transformer model architecture
            design principles and components"

        ADVERSARIAL:
            Target: Missed comparative content. Some relevant chunks discuss
            the topic in contrast to something else ("unlike RNNs, attention...").
            Adversarial framing surfaces these comparison-style passages.
            Example: "how attention works" -> "how transformers differ from
            recurrent models in processing sequence information"

    All five queries retrieve independently. Results are fused via RRF with
    equal weights (w_i = 1/5 = 0.2), producing a single ranked list where
    documents retrieved by MULTIPLE perspectives score highest.

    Args:
        question    : User query.
        vectorstore : ChromaDB.
        llm_expand  : LLM with expansion temperature=0.3.
        llm_answer  : LLM with answer temperature=0.0.
        config      : QEConfig.

    Returns:
        QETrace with 5 typed perspective queries and RRF-fused results.
    """
    trace = QETrace(original_query=question, strategy="Config3_5Perspectives_RRF")
    t0    = time.perf_counter()

    # Generate 5 perspective reformulations
    t_expand   = time.perf_counter()
    exp_prompt = ChatPromptTemplate.from_template(PERSPECTIVE_EXPANSION_PROMPT)
    raw_output = (exp_prompt | llm_expand | StrOutputParser()).invoke({
        "question": question,
        "n_views":  config.N_EXPANSIONS,
    })

    # Parse: one question per line, skip empty/short lines
    expanded = [
        line.strip() for line in raw_output.strip().split("\n")
        if line.strip() and len(line.strip()) > 10
    ][: config.N_EXPANSIONS]

    # Always include original query as 6th retrieval path
    all_queries = [question] + expanded
    trace.expanded_queries = all_queries
    trace.expansion_ms     = (time.perf_counter() - t_expand) * 1000

    logger.info(
        "Config3: generated %d perspective queries + original = %d total",
        len(expanded), len(all_queries),
    )
    for i, q in enumerate(all_queries, 1):
        logger.info("  [%d] %s", i, q)

    # Retrieve for each query independently
    t_ret     = time.perf_counter()
    doc_lists: List[List[Document]] = []

    for q in all_queries:
        docs = vectorstore.similarity_search(q, k=config.SEARCH_K)
        doc_lists.append(docs)
        trace.docs_per_query.append(len(docs))

    trace.retrieval_ms   = (time.perf_counter() - t_ret) * 1000
    trace.total_candidates = sum(trace.docs_per_query)

    # RRF fusion: equal weights across all queries
    fused_docs = rrf_fusion(
        doc_lists,
        weights=None,     # equal weights 1/N
        k=config.RRF_K,
        final_k=config.FINAL_K,
        prefix_len=config.DEDUP_PREFIX_LEN,
    )
    trace.final_docs = fused_docs

    context = format_docs(fused_docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    answer  = (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [96]:


# ---------------------------------------------------------------------------
# CONFIG 4: Step-Back Prompting + Original Query Fusion
# ---------------------------------------------------------------------------

STEP_BACK_PROMPT = """You are a retrieval optimization expert.
Given a specific question, generate a more abstract, general "step-back" question
that asks about the underlying concept, principle, or broader topic.
The step-back question should retrieve BACKGROUND CONTEXT and FOUNDATIONAL PRINCIPLES
that help ground the answer to the specific question.

Rules:
- The step-back question should be more general/abstract than the original.
- It must be relevant to understanding the original question.
- Output ONLY the step-back question, no explanation.

Specific question: {question}

Step-back question (abstract, principle-level):"""

STEPBACK_RAG_PROMPT = """You are a precise technical research assistant.
Answer the specific question using the context below.
The context contains both specific evidence (from the original question)
and background principles (from the step-back question).
Use ONLY information from the context.

Context:
{context}

Specific question: {question}

Answer:"""


def run_config4_step_back(
    question:    str,
    vectorstore: Chroma,
    llm_expand:  ChatOllama,
    llm_answer:  ChatOllama,
    config:      QEConfig,
) -> QETrace:
    """
    Configuration 4: Step-Back Prompting + Original Query Fusion.

    Source: This paper uses an LLM to generate a "step back" question.
    With retrieval, both the "step back" question and the original question are
    used to do retrieval, and then both results are used to ground the language
    model response.

    Step-Back Design:
        The step-back question abstracts the original query to a PRINCIPLE level.
        This retrieves background context passages (typically introduction sections,
        overview paragraphs, and concept definitions) that:
            1. Provide the LLM with conceptual grounding for its answer.
            2. Capture relevant content that uses different vocabulary than the
               specific question (principle-level language vs. instance-level).
            3. Retrieve passages that would not surface from the specific query
               but are necessary to understand the answer context.

    Example step-back transformations:
        Original:  "What BLEU score did the Transformer achieve on WMT 2014 EN-DE?"
        Step-back: "What evaluation metrics are used for machine translation models?"

        Original:  "How does BERT use the [MASK] token during pre-training?"
        Step-back: "What are the general principles of masked language model pre-training?"

    Fusion strategy:
        Step-back retrieves SEARCH_K docs (background context).
        Original retrieves SEARCH_K docs (specific evidence).
        RRF fusion with weights [0.4 step-back, 0.6 original] biases toward
        the specific evidence while preserving background context coverage.
        The weight asymmetry reflects the production observation that step-back
        context is SUPPLEMENTARY (needs less representation) while original
        evidence is PRIMARY (needs more representation) in the answer context.

    Args:
        question    : Specific user query.
        vectorstore : ChromaDB.
        llm_expand  : LLM for step-back generation (temperature=0.0 for consistency).
        llm_answer  : LLM for answer generation (temperature=0.0).
        config      : QEConfig.

    Returns:
        QETrace with original + step-back queries and fused context.
    """
    trace = QETrace(original_query=question, strategy="Config4_StepBack_Fusion")
    t0    = time.perf_counter()

    # Generate step-back question (deterministic: temperature=0.0)
    t_expand    = time.perf_counter()
    sb_prompt   = ChatPromptTemplate.from_template(STEP_BACK_PROMPT)
    step_back_q = (sb_prompt | llm_expand | StrOutputParser()).invoke(
        {"question": question}
    ).strip()
    trace.expansion_ms     = (time.perf_counter() - t_expand) * 1000
    trace.expanded_queries = [question, step_back_q]
    logger.info("Step-Back: '%s'", step_back_q)

    # Retrieve for original and step-back independently
    t_ret = time.perf_counter()
    orig_docs    = vectorstore.similarity_search(question,   k=config.SEARCH_K)
    stepback_docs = vectorstore.similarity_search(step_back_q, k=config.SEARCH_K)
    trace.retrieval_ms   = (time.perf_counter() - t_ret) * 1000
    trace.docs_per_query = [len(orig_docs), len(stepback_docs)]
    trace.total_candidates = len(orig_docs) + len(stepback_docs)

    # RRF fusion: bias toward original query (0.6) over step-back (0.4)
    fused_docs = rrf_fusion(
        [orig_docs, stepback_docs],
        weights=[0.6, 0.4],     # original query has higher weight
        k=config.RRF_K,
        final_k=config.FINAL_K,
        prefix_len=config.DEDUP_PREFIX_LEN,
    )
    trace.final_docs = fused_docs

    context = format_docs(fused_docs)
    prompt  = ChatPromptTemplate.from_template(STEPBACK_RAG_PROMPT)
    answer  = (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [97]:

# ---------------------------------------------------------------------------
# CONFIG 5: RFG -- Retrieval-Feedback-Grounded Expansion [BEST]
# ---------------------------------------------------------------------------

RFG_FEEDBACK_PROMPT = """You are a query expansion expert.
Given:
1. The original user question
2. A set of initially retrieved document passages (pseudo-relevant feedback)

Generate {n_expansions} GROUNDED query reformulations that are:
- Based on vocabulary and concepts FOUND IN the retrieved passages (not invented)
- Targeting aspects of the question not well-covered by the initial retrieval
- Specific enough to retrieve additional relevant passages
- Different from each other and from the original question

Also extract a comma-separated list of domain-specific synonyms and acronyms
found in the passages that are relevant to the question.

Output format (strict):
QUERIES:
<one reformulated query per line, {n_expansions} total>
SYNONYMS:
<comma-separated technical synonyms and acronym expansions>

Original question: {question}

Initially retrieved passages:
{feedback_context}

Output:"""


def run_config5_rfg_expansion(
    question:    str,
    vectorstore: Chroma,
    llm_expand:  ChatOllama,
    llm_answer:  ChatOllama,
    config:      QEConfig,
) -> QETrace:
    """
    Configuration 5: RFG -- Retrieval-Feedback-Grounded Query Expansion [BEST].

    RFG addresses the fundamental weakness of all expansion approaches in Configs
    2-4: they rely on the LLM's PARAMETRIC KNOWLEDGE to generate expansions. When
    the corpus domain differs from the LLM's training data, parametric expansion
    hallucinates vocabulary that does not appear in the actual index, producing
    queries that retrieve irrelevant content.

    The RFG framework uses pseudo-relevance feedback for an initial
    retrieval step and generates pseudo-queries based on the retrieved information.
    This significantly improves information retrieval by avoiding hallucinations,
    achieving better results than simply generating multiple queries.

    RFG PIPELINE:

        Stage 1 -- Initial Retrieval (pseudo-relevance feedback):
            Execute the ORIGINAL question against the index.
            Retrieve RFG_INITIAL_K=10 documents (the "pseudo-relevant" set).
            These documents ANCHOR all subsequent expansion in corpus vocabulary.

        Stage 2 -- Grounded Expansion Generation:
            Pass the original question AND the 10 pseudo-relevant passages to the LLM.
            LLM generates N reformulations using ONLY vocabulary and concepts found
            in the actual retrieved passages. This grounding prevents hallucination.
            LLM also extracts domain synonyms and acronym expansions from the passages.

        Stage 3 -- Synonym/Acronym Query Augmentation:
            Acronyms and abbreviations are common in queries and documents.
            "LLM" should ideally match documents mentioning "Large Language Model."
            The extracted synonyms are used to create N+1 additional augmented queries:
            original_query + " " + synonym_group (e.g., "attention mechanism
            query-key-value pairs scaled dot-product").

        Stage 4 -- Full Expansion Retrieval + RRF Fusion:
            Retrieve for: original + N grounded expansions + synonym-augmented.
            Total query count: 1 + N + 1 = N+2 queries.
            RRF fusion with equal weights produces the final ranked document list.

    WHY THIS IS THE BEST CONFIGURATION:
        - Grounded in corpus vocabulary: zero hallucinated queries.
        - Synonym expansion covers acronym mismatch (MLM -> Masked Language Model).
        - RFG retrieves more unique relevant documents per query than any
          parametric expansion approach on domain-specific corpora.
        - The pseudo-relevant feedback loop adapts automatically to any corpus
          without any domain-specific prompt engineering.

    Args:
        question    : User query.
        vectorstore : ChromaDB.
        llm_expand  : LLM with expansion temperature=0.3.
        llm_answer  : LLM with answer temperature=0.0.
        config      : QEConfig.

    Returns:
        QETrace with grounded RFG expansions, synonym augmentation, and RRF fusion.
    """
    trace = QETrace(original_query=question, strategy="Config5_RFG_SynonymExpansion [BEST]")
    t0    = time.perf_counter()

    # Stage 1: Initial retrieval (pseudo-relevance feedback anchoring)
    t_ret_init   = time.perf_counter()
    initial_docs = vectorstore.similarity_search(question, k=config.RFG_INITIAL_K)
    logger.info("RFG Stage 1: retrieved %d pseudo-relevant docs", len(initial_docs))

    # Build feedback context (first 250 chars per doc to stay within LLM budget)
    feedback_context = "\n\n".join([
        f"[Passage {i+1} | {d.metadata.get('paper_name','?')[:25]} p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()[:250]}"
        for i, d in enumerate(initial_docs)
    ])

    # Stage 2: Grounded expansion generation
    t_expand   = time.perf_counter()
    rfg_prompt = ChatPromptTemplate.from_template(RFG_FEEDBACK_PROMPT)
    raw_output = (rfg_prompt | llm_expand | StrOutputParser()).invoke({
        "question":         question,
        "feedback_context": feedback_context,
        "n_expansions":     config.N_EXPANSIONS,
    })
    trace.expansion_ms = (time.perf_counter() - t_expand) * 1000

    # Parse QUERIES and SYNONYMS sections from LLM output
    grounded_queries: List[str] = []
    synonyms_str:     str       = ""

    try:
        if "QUERIES:" in raw_output and "SYNONYMS:" in raw_output:
            q_section, s_section = raw_output.split("SYNONYMS:", 1)
            q_lines = q_section.replace("QUERIES:", "").strip().split("\n")
            grounded_queries = [
                l.strip() for l in q_lines
                if l.strip() and len(l.strip()) > 10
            ][: config.N_EXPANSIONS]
            synonyms_str = s_section.strip().replace("\n", ", ")
        else:
            # Fallback: treat all lines as queries (no synonyms section found)
            grounded_queries = [
                l.strip() for l in raw_output.strip().split("\n")
                if l.strip() and len(l.strip()) > 10
            ][: config.N_EXPANSIONS]
    except Exception as exc:
        logger.warning("RFG output parse failed (%s). Using raw lines.", exc)
        grounded_queries = [
            l.strip() for l in raw_output.strip().split("\n")
            if l.strip() and len(l.strip()) > 10
        ][: config.N_EXPANSIONS]

    logger.info("RFG Stage 2: %d grounded expansions generated", len(grounded_queries))
    logger.info("RFG synonyms extracted: '%s'", synonyms_str[:100])

    # Stage 3: Synonym-augmented query
    # Combine original question + top synonyms for a vocabulary-enriched retrieval
    synonym_augmented_q = question
    if synonyms_str:
        # Limit synonym extension to 50 chars to avoid over-augmentation
        synonym_terms = [s.strip() for s in synonyms_str.split(",") if s.strip()][:5]
        synonym_augmented_q = question + " " + " ".join(synonym_terms)
    logger.info("Synonym-augmented query: '%s'", synonym_augmented_q[:100])

    # Stage 4: Retrieve for all queries
    all_queries = [question] + grounded_queries + [synonym_augmented_q]
    trace.expanded_queries = all_queries

    t_ret  = time.perf_counter()
    doc_lists: List[List[Document]] = []

    for q in all_queries:
        docs = vectorstore.similarity_search(q, k=config.SEARCH_K)
        doc_lists.append(docs)
        trace.docs_per_query.append(len(docs))

    trace.retrieval_ms     = (time.perf_counter() - t_ret + time.perf_counter() - t_ret_init) * 1000
    trace.total_candidates = sum(trace.docs_per_query)

    # RRF fusion: equal weights
    fused_docs = rrf_fusion(
        doc_lists,
        weights=None,   # equal weights
        k=config.RRF_K,
        final_k=config.FINAL_K,
        prefix_len=config.DEDUP_PREFIX_LEN,
    )
    trace.final_docs = fused_docs

    context = format_docs(fused_docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    answer  = (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [98]:

# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:    str,
    vectorstore: Chroma,
    llm_expand:  ChatOllama,
    llm_stepback: ChatOllama,
    llm_answer:  ChatOllama,
    config:      QEConfig,
) -> Dict[str, QETrace]:
    """Run all five query expansion configurations on a single question."""
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = [
        ("Config1_Baseline",
         lambda q: run_config1_baseline(q, vectorstore, llm_answer, config)),
        ("Config2_MultiQuery_BuiltIn",
         lambda q: run_config2_multi_query_builtin(q, vectorstore, llm_expand, llm_answer, config)),
        ("Config3_5Perspectives_RRF",
         lambda q: run_config3_perspective_expansion(q, vectorstore, llm_expand, llm_answer, config)),
        ("Config4_StepBack_Fusion",
         lambda q: run_config4_step_back(q, vectorstore, llm_stepback, llm_answer, config)),
        ("Config5_RFG_Grounded [BEST]",
         lambda q: run_config5_rfg_expansion(q, vectorstore, llm_expand, llm_answer, config)),
    ]

    traces: Dict[str, QETrace] = {}

    for label, fn in runners:
        print(f"\n{'='*60}\nRUNNING: {label}\n{'='*60}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    # Latency summary table
    print("\n" + "=" * 78)
    print("QUERY EXPANSION LATENCY COMPARISON")
    print(f"{'Config':<44} {'Queries':>8} {'Cands':>6} "
          f"{'Exp(ms)':>8} {'Ret(ms)':>8} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<44} {len(tr.expanded_queries):>8} "
            f"{tr.total_candidates:>6} "
            f"{tr.expansion_ms:>8.1f} "
            f"{tr.retrieval_ms:>8.1f} "
            f"{tr.total_ms:>10.1f}"
        )
    print("=" * 78 + "\n")

    return traces


In [99]:
"""
    End-to-end Query Expansion RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs.
        2.  Load and chunk documents.
        3.  Validate Azure credentials.
        4.  Initialize three LLM instances:
                llm_expand:   temperature=0.3 (creative expansion)
                llm_stepback: temperature=0.0 (deterministic abstraction)
                llm_answer:   temperature=0.0 (deterministic answer)
        5.  Load BGE bi-encoder.
        6.  Build/load ChromaDB index.
        7.  Run all five configurations on demo queries.

    LLM CALL COUNT PER QUERY (approximate):
        Config 1: 1 call  (answer only)
        Config 2: 2 calls (1 expansion + 1 answer)
        Config 3: 2 calls (1 expansion generating 5 queries + 1 answer)
        Config 4: 2 calls (1 step-back generation + 1 answer)
        Config 5: 2 calls (1 RFG expansion over 10 docs + 1 answer)

    All expansion configs = exactly 2 LLM calls per query.
    The difference is in the QUALITY and GROUNDING of the expansion.
    """

'\n    End-to-end Query Expansion RAG pipeline orchestrator.\n\n    Execution order:\n        1.  Download three arXiv PDFs.\n        2.  Load and chunk documents.\n        3.  Validate Azure credentials.\n        4.  Initialize three LLM instances:\n                llm_expand:   temperature=0.3 (creative expansion)\n                llm_stepback: temperature=0.0 (deterministic abstraction)\n                llm_answer:   temperature=0.0 (deterministic answer)\n        5.  Load BGE bi-encoder.\n        6.  Build/load ChromaDB index.\n        7.  Run all five configurations on demo queries.\n\n    LLM CALL COUNT PER QUERY (approximate):\n        Config 1: 1 call  (answer only)\n        Config 2: 2 calls (1 expansion + 1 answer)\n        Config 3: 2 calls (1 expansion generating 5 queries + 1 answer)\n        Config 4: 2 calls (1 step-back generation + 1 answer)\n        Config 5: 2 calls (1 RFG expansion over 10 docs + 1 answer)\n\n    All expansion configs = exactly 2 LLM calls per query

In [100]:
config = QEConfig()
logger.info("=== Query Expansion RAG Pipeline Starting ===")


2026-05-20 23:11:06 | INFO     | query_expansion_rag | === Query Expansion RAG Pipeline Starting ===


In [101]:
pdf_paths  = download_pdfs(config)
chunks     = load_and_chunk_documents(pdf_paths, config)


2026-05-20 23:11:06 | INFO     | query_expansion_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-20 23:11:06 | INFO     | query_expansion_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-20 23:11:06 | INFO     | query_expansion_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)
2026-05-20 23:11:14 | INFO     | query_expansion_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-20 23:11:16 | INFO     | query_expansion_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-20 23:11:16 | INFO     | query_expansion_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-20 23:11:16 | INFO     | query_expansion_rag | Total chunks: 458


In [102]:

llm_expand   = build_ollama_llm(config, temperature=config.LLM_EXPANSION_TEMPERATURE)
llm_stepback = build_ollama_llm(config, temperature=config.LLM_STEPBACK_TEMPERATURE)
llm_answer   = build_ollama_llm(config, temperature=config.LLM_ANSWER_TEMPERATURE)


In [103]:
embeddings  = build_bge_embeddings(config)


2026-05-20 23:11:23 | INFO     | query_expansion_rag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-20 23:11:23 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-20 23:11:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 23:11:24 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-20 23:11:24 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 23:11:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3558.50it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-20 23:11:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 23:11:28 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-20 23:11:29 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 23:11:29 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-20 23:11:29 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-20 23:11:29 | INFO     | httpx |

In [104]:
vectorstore = build_chroma_index(chunks, embeddings, config)


2026-05-20 23:11:31 | INFO     | query_expansion_rag | Loading ChromaDB from './chroma_qe_db' (458 vectors) ...


In [105]:
demo_questions = [
        # Vocabulary mismatch -- synonym expansion critical
        "How does the attention scoring function compute weights over the input?",

        # Expertise asymmetry -- expert-vocabulary expansion critical
        "What training approach does BERT use to learn from unlabeled text?",

        # Specificity mismatch -- narrow and broad scope both needed
        "How many encoder and decoder layers are in the Transformer base model?",

        # Step-back test -- abstract principle retrieval aids answer grounding
        "What BLEU score did the Transformer achieve on WMT 2014 English-German translation?",

        # RFG test -- acronym expansion critical for corpus-specific terminology
        "How does the MLM pre-training objective in BERT prevent the model from trivially predicting masked tokens?",
    ]


In [106]:

for question in demo_questions:
    run_all_configs(question, vectorstore, llm_expand, llm_stepback, llm_answer, config)

logger.info("=== Query Expansion RAG Pipeline Demo Complete ===")




##############################################################################
QUERY: How does the attention scoring function compute weights over the input?
##############################################################################

RUNNING: Config1_Baseline
2026-05-20 23:11:46 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline
Query: How does the attention scoring function compute weights over the input?

  Expanded Queries (1):
  [1] (4 docs)  How does the attention scoring function compute weights over the input?

  Candidates: 4  ->  Final: 4
  Sources: [Attention Is All You Need p3] [Attention Is All You Need p9] [Attention Is All You Need p2] [Attention Is All You Need p3] 

  Latency: expansion=0.0ms  retrieval=835.8ms  total=22456.8ms

  ANSWER:
  The attention scoring function computes weights over the input by taking the dot product of each query with all keys, dividing each result by √dk, and then applying

In [107]:
smoke_query = "How does attention work in transformers?"
smoke_traces = run_all_configs(smoke_query, vectorstore, llm_expand, llm_stepback, llm_answer, config)
print("Successful configs:", list(smoke_traces.keys()))
print("Count:", len(smoke_traces))


##############################################################################
QUERY: How does attention work in transformers?
##############################################################################

RUNNING: Config1_Baseline
2026-05-20 23:29:04 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline
Query: How does attention work in transformers?

  Expanded Queries (1):
  [1] (4 docs)  How does attention work in transformers?

  Candidates: 4  ->  Final: 4
  Sources: [Attention Is All You Need p4] [Attention Is All You Need p9] [Attention Is All You Need p1] [Attention Is All You Need p0] 

  Latency: expansion=0.0ms  retrieval=271.4ms  total=13934.4ms

  ANSWER:
  In transformers, attention mechanisms allow each position in a sequence (such as a word in a sentence) to attend to all other positions in the same sequence, enabling the model to weigh the importance of different parts of the input when making predictions.